# 16. Stage 1 Golden Set Quality Audit

현재 낮은 LLM 성능 중 일부가 모델 오류가 아니라 Golden Set과 Prediction의 표현 방식 차이에서 발생했는지 사람이 짧게 확인할 QA 후보를 만든다.

이 Notebook은 정답을 수정하거나 성능을 다시 계산하지 않는다. 오직 검토 후보를 추출하고 정리한다.


## 0. QA 목적과 금지사항

- Golden Set v1.0 및 기존 Prediction/Evaluation 파일을 수정하거나 덮어쓰지 않는다.
- LLM Prediction을 정답으로 간주하지 않고 모델 결과에 맞춰 Gold를 바꾸지 않는다.
- Current Rule, Extended Rule, LLM 성능을 다시 계산하지 않는다.
- Prompt와 Rule을 수정하지 않는다.
- 설문 Query를 읽지 않는다.
- 200개 전체 재검수를 만들지 않는다.
- Embedding, LLM, 형태소 분석, 의미 유사도는 사용하지 않는다.
- 자동 비교는 JSON parse, exact/소문자/공백/중복/순서/개수 차이만 허용한다.


## 1. Imports / Paths


In [1]:
import collections
import hashlib
import json
import pathlib
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = pathlib.Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
GOLDSET_PATH = PROJECT_ROOT / "evaluation_data" / "stage1" / "13_stage1_golden_set_v1_200.xlsx"
EXT_PRED_PATH = OUTPUT_DIR / "14_extended_rule_stage1_predictions.csv"
EXT_EVAL_PATH = OUTPUT_DIR / "14_extended_rule_stage1_evaluation.csv"
LLM_PRED_PATH = OUTPUT_DIR / "15_llm_stage1_predictions.csv"
LLM_EVAL_PATH = OUTPUT_DIR / "15_llm_stage1_evaluation.csv"
QUERY_CHANGES_PATH = OUTPUT_DIR / "15_llm_vs_extended_query_changes.csv"
LLM_ERRORS_PATH = OUTPUT_DIR / "15_llm_stage1_errors.csv"

CANDIDATES_PATH = OUTPUT_DIR / "16_golden_set_quality_audit_candidates.csv"
GUIDE_PATH = OUTPUT_DIR / "16_golden_set_quality_audit_guide.md"
RANDOM_SEED = 42
ADDITIONAL_SAMPLE_SIZE = 20

INPUT_PATHS = {
    "Golden Set": GOLDSET_PATH,
    "Extended predictions": EXT_PRED_PATH,
    "Extended evaluation": EXT_EVAL_PATH,
    "LLM predictions": LLM_PRED_PATH,
    "LLM evaluation": LLM_EVAL_PATH,
    "LLM vs Extended changes": QUERY_CHANGES_PATH,
    "LLM errors": LLM_ERRORS_PATH,
}
for label, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"필수 입력 파일이 없습니다 - {label}: {path}")
    if re.search(r"\s*\(\d+\)$", path.stem):
        raise ValueError(f"복제 suffix가 붙은 파일은 사용하지 않습니다: {path.name}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_hashes_before = {label: sha256_file(path) for label, path in INPUT_PATHS.items()}
display(pd.DataFrame([{"input": label, "file": path.name, "sha256": input_hashes_before[label]} for label, path in INPUT_PATHS.items()]))


,input,file,sha256
0,Golden Set,13_stage1_golden_set_v1_200.xlsx,a56613ad4bb082d2a05408f0a888727945cbc924a33ca0...
1,Extended predictions,14_extended_rule_stage1_predictions.csv,c5edafa792b8ce74cf32a52378975ec15759e47ce74e5a...
2,Extended evaluation,14_extended_rule_stage1_evaluation.csv,94e044d4e30b69a6d8b5e8e5e88dfaf5df45e3310198b2...
3,LLM predictions,15_llm_stage1_predictions.csv,f36e7266f2319be30cb8ec67f7c847f2d7283815accf0d...
4,LLM evaluation,15_llm_stage1_evaluation.csv,0456d5a7e6f5ba7319ca9622f6683eee214d1f84a582ac...
5,LLM vs Extended changes,15_llm_vs_extended_query_changes.csv,8d41f9d04379b4d45ac52aaf88dd4f26e1efc53e959e18...
6,LLM errors,15_llm_stage1_errors.csv,606e2d37c64346e11d548952a0387e59238ab59cdee916...


## 2. Load Golden Set


In [2]:
GOLD_JSON_COLUMNS = ["gold_scent_preference", "gold_context", "gold_performance", "gold_avoid", "gold_additional_requirements"]
GOLD_COLUMNS = ["query_id", "query_text", *GOLD_JSON_COLUMNS]

def parse_json_cell(value, query_id, column):
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f"JSON parsing 실패: query_id={query_id}, column={column}") from exc

gold_raw_df = pd.read_excel(GOLDSET_PATH, sheet_name="Golden Set", header=4, dtype=str, keep_default_na=False)
if gold_raw_df.columns.tolist() != GOLD_COLUMNS or len(gold_raw_df) != 200:
    raise ValueError(f"Golden Set schema/row 오류: columns={gold_raw_df.columns.tolist()}, rows={len(gold_raw_df)}")
gold_df = gold_raw_df.copy()
for column in GOLD_JSON_COLUMNS:
    gold_df[column] = [parse_json_cell(value, query_id, column) for value, query_id in zip(gold_raw_df[column], gold_raw_df["query_id"])]
if gold_df["query_id"].duplicated().any() or gold_df["query_text"].duplicated().any():
    raise ValueError("Golden Set query_id/query_text 중복")
print("Golden Set rows:", len(gold_df))


Golden Set rows: 200


## 3. Load Extended Predictions


In [3]:
EXT_PRED_COLUMNS = ["query_id", "query_text", "pred_scent_preference", "pred_context", "pred_performance", "pred_avoid", "pred_additional_requirements"]
ext_pred_raw_df = pd.read_csv(EXT_PRED_PATH, dtype=str, keep_default_na=False)
ext_eval_df = pd.read_csv(EXT_EVAL_PATH, dtype=str, keep_default_na=False)
if ext_pred_raw_df.columns.tolist() != EXT_PRED_COLUMNS or len(ext_pred_raw_df) != 200 or len(ext_eval_df) != 200:
    raise ValueError("Extended Prediction/Evaluation schema 또는 row 수 오류")
ext_pred_df = ext_pred_raw_df.copy()
for column in EXT_PRED_COLUMNS[2:]:
    ext_pred_df[column] = [parse_json_cell(value, query_id, column) for value, query_id in zip(ext_pred_raw_df[column], ext_pred_raw_df["query_id"])]
print("Extended rows:", len(ext_pred_df), len(ext_eval_df))


Extended rows: 200 200


## 4. Load LLM Predictions


In [4]:
LLM_PRED_JSON_COLUMNS = ["pred_scent_preference", "pred_context", "pred_performance", "pred_avoid", "pred_additional_requirements"]
llm_pred_raw_df = pd.read_csv(LLM_PRED_PATH, dtype=str, keep_default_na=False)
llm_eval_df = pd.read_csv(LLM_EVAL_PATH, dtype=str, keep_default_na=False)
query_changes_df = pd.read_csv(QUERY_CHANGES_PATH, dtype=str, keep_default_na=False)
llm_errors_df = pd.read_csv(LLM_ERRORS_PATH, dtype=str, keep_default_na=False)
if len(llm_pred_raw_df) != 200 or len(llm_eval_df) != 200 or len(query_changes_df) != 200:
    raise ValueError("LLM Prediction/Evaluation/Change row 수는 각각 200이어야 합니다.")
llm_pred_df = llm_pred_raw_df.copy()
for column in LLM_PRED_JSON_COLUMNS:
    llm_pred_df[column] = [parse_json_cell(value, query_id, column) for value, query_id in zip(llm_pred_raw_df[column], llm_pred_raw_df["query_id"])]
print("LLM rows:", len(llm_pred_df), len(llm_eval_df), "LLM error rows:", len(llm_errors_df))


LLM rows: 200 200 LLM error rows: 188


## 5. Validate Query Alignment

네 데이터셋의 200개 `query_id`와 `query_text`가 Golden Set과 정확히 일치하는지 확인한다.


In [5]:
def query_pairs(frame):
    return list(frame[["query_id", "query_text"]].itertuples(index=False, name=None))

gold_pairs = query_pairs(gold_df)
alignment_frames = {
    "Extended predictions": ext_pred_df,
    "Extended evaluation": ext_eval_df,
    "LLM predictions": llm_pred_df,
    "LLM evaluation": llm_eval_df,
    "LLM vs Extended changes": query_changes_df,
}
alignment_rows = []
for label, frame in alignment_frames.items():
    aligned = query_pairs(frame) == gold_pairs
    alignment_rows.append({"source": label, "rows": len(frame), "unique_query_id": frame["query_id"].nunique(), "exact_order_alignment": aligned})
    if len(frame) != 200 or frame["query_id"].nunique() != 200 or not aligned:
        raise ValueError(f"Query alignment 실패: {label}")
if set(llm_errors_df["query_id"]) != set(llm_eval_df.loc[llm_eval_df["overall_exact_match"].str.casefold().ne("true"), "query_id"]):
    raise ValueError("LLM errors와 LLM evaluation 실패 Query 집합 불일치")
display(pd.DataFrame(alignment_rows))


,source,rows,unique_query_id,exact_order_alignment
0,Extended predictions,200,200,True
1,Extended evaluation,200,200,True
2,LLM predictions,200,200,True
3,LLM evaluation,200,200,True
4,LLM vs Extended changes,200,200,True


## 6. Extract LLM_REGRESSED

Extended Rule 전체 Exact Match 성공, LLM 실패인 Query를 실제 change 파일에서 전부 추출한다. 예상 개수를 하드코딩하지 않는다.


In [6]:
llm_regressed_ids = set(query_changes_df.loc[query_changes_df["change_type"].eq("LLM_REGRESSED"), "query_id"])
llm_regressed_df = query_changes_df.loc[query_changes_df["query_id"].isin(llm_regressed_ids)].copy()
print("LLM_REGRESSED count:", len(llm_regressed_df))


LLM_REGRESSED count: 5


## 7. Extract Avoid Mismatch

Gold Avoid가 비어 있지 않고 기존 LLM 평가의 Avoid match가 실패한 Query를 전부 포함한다. 표현이 의미상 같은지는 자동 판정하지 않는다.


In [7]:
def csv_bool(value):
    return str(value).strip().casefold() == "true"

gold_avoid_by_id = gold_df.set_index("query_id")["gold_avoid"]
avoid_mismatch_ids = set(
    llm_eval_df.loc[
        llm_eval_df["query_id"].map(lambda qid: bool(gold_avoid_by_id.loc[qid]))
        & ~llm_eval_df["avoid_match"].map(csv_bool),
        "query_id",
    ]
)
avoid_mismatch_df = llm_eval_df.loc[llm_eval_df["query_id"].isin(avoid_mismatch_ids)].copy()
print("Avoid mismatch count:", len(avoid_mismatch_df))


Avoid mismatch count: 36


## 8. Sample Additional Mismatch

Gold Additional이 존재하고 LLM Additional과 일치하지 않는 Query에서 A/B 그룹을 제외한 뒤 `random_state=42`로 최대 20개만 뽑는다.


In [8]:
gold_additional_by_id = gold_df.set_index("query_id")["gold_additional_requirements"]
excluded_ids = llm_regressed_ids | avoid_mismatch_ids
additional_pool_df = llm_eval_df.loc[
    llm_eval_df["query_id"].map(lambda qid: bool(gold_additional_by_id.loc[qid]))
    & ~llm_eval_df["additional_match"].map(csv_bool)
    & ~llm_eval_df["query_id"].isin(excluded_ids)
].copy()
sample_n = min(ADDITIONAL_SAMPLE_SIZE, len(additional_pool_df))
additional_sample_df = additional_pool_df.sample(n=sample_n, random_state=RANDOM_SEED).sort_values("query_id").reset_index(drop=True)
additional_sample_ids = set(additional_sample_df["query_id"])
print("Additional mismatch pool after exclusions:", len(additional_pool_df))
print("Additional sample count:", len(additional_sample_df))


Additional mismatch pool after exclusions: 139
Additional sample count: 20


## 9. Merge / Deduplicate QA Candidates


In [9]:
gold_by_id = gold_df.set_index("query_id")
ext_pred_by_id = ext_pred_df.set_index("query_id")
llm_pred_by_id = llm_pred_df.set_index("query_id")
ext_eval_by_id = ext_eval_df.set_index("query_id")
llm_eval_by_id = llm_eval_df.set_index("query_id")

selected_ids = [qid for qid in gold_df["query_id"] if qid in llm_regressed_ids | avoid_mismatch_ids | additional_sample_ids]
candidate_rows = []
for query_id in selected_ids:
    reasons = []
    if query_id in llm_regressed_ids: reasons.append("LLM_REGRESSED")
    if query_id in avoid_mismatch_ids: reasons.append("AVOID_MISMATCH")
    if query_id in additional_sample_ids: reasons.append("ADDITIONAL_SAMPLE")
    gold = gold_by_id.loc[query_id]
    ext = ext_pred_by_id.loc[query_id]
    llm = llm_pred_by_id.loc[query_id]
    candidate_rows.append({
        "query_id": query_id,
        "query_text": gold.query_text,
        "selection_reason": "|".join(reasons),
        "gold_scent": gold.gold_scent_preference,
        "gold_context": gold.gold_context,
        "gold_performance": gold.gold_performance,
        "gold_avoid": gold.gold_avoid,
        "gold_additional": gold.gold_additional_requirements,
        "extended_scent": ext.pred_scent_preference,
        "extended_context": ext.pred_context,
        "extended_performance": ext.pred_performance,
        "extended_avoid": ext.pred_avoid,
        "extended_additional": ext.pred_additional_requirements,
        "llm_scent": llm.pred_scent_preference,
        "llm_context": llm.pred_context,
        "llm_performance": llm.pred_performance,
        "llm_avoid": llm.pred_avoid,
        "llm_additional": llm.pred_additional_requirements,
        "extended_exact": csv_bool(ext_eval_by_id.loc[query_id, "overall_exact_match"]),
        "llm_exact": csv_bool(llm_eval_by_id.loc[query_id, "overall_exact_match"]),
        "avoid_exact": csv_bool(llm_eval_by_id.loc[query_id, "avoid_match"]),
        "additional_exact": csv_bool(llm_eval_by_id.loc[query_id, "additional_match"]),
    })
candidates_df = pd.DataFrame(candidate_rows)
if candidates_df["query_id"].duplicated().any():
    raise RuntimeError("QA 후보 중복 제거 실패")
print("Unique audit queries:", len(candidates_df))


Unique audit queries: 60


## 10. Add Mechanical Comparison Columns

정규화는 lowercase, strip, 중복 제거, 순서 무시까지만 적용한다. 결과는 검토 보조 정보이며 의미 동일성을 뜻하지 않는다.


In [10]:
def normalize_items(values):
    return frozenset(str(value).strip().casefold() for value in values if str(value).strip())

def normalized_equal(left, right):
    return normalize_items(left) == normalize_items(right)

candidates_df["avoid_normalized_equal"] = [normalized_equal(gold, pred) for gold, pred in zip(candidates_df["gold_avoid"], candidates_df["llm_avoid"])]
candidates_df["additional_normalized_equal"] = [normalized_equal(gold, pred) for gold, pred in zip(candidates_df["gold_additional"], candidates_df["llm_additional"])]
candidates_df["avoid_item_count_gold"] = candidates_df["gold_avoid"].map(len)
candidates_df["avoid_item_count_llm"] = candidates_df["llm_avoid"].map(len)
candidates_df["additional_item_count_gold"] = candidates_df["gold_additional"].map(len)
candidates_df["additional_item_count_llm"] = candidates_df["llm_additional"].map(len)

def diff_fields(query_id):
    row = llm_eval_by_id.loc[query_id]
    mapping = [("scent", "scent_match"), ("context", "context_match"), ("performance", "performance_match"), ("avoid", "avoid_match"), ("additional", "additional_match")]
    return "|".join(field for field, column in mapping if not csv_bool(row[column]))

candidates_df["diff_fields"] = candidates_df["query_id"].map(diff_fields)
candidates_df["manual_decision"] = ""
candidates_df["manual_issue_type"] = ""
candidates_df["manual_comment"] = ""


## 11. Display LLM_REGRESSED


In [11]:
regressed_columns = [
    "query_id", "query_text", "gold_scent", "llm_scent", "gold_context", "llm_context",
    "gold_performance", "llm_performance", "gold_avoid", "llm_avoid",
    "gold_additional", "llm_additional", "diff_fields",
]
llm_regressed_view = candidates_df.loc[candidates_df["query_id"].isin(llm_regressed_ids), regressed_columns]
display(llm_regressed_view)


,query_id,query_text,gold_scent,llm_scent,gold_context,llm_context,gold_performance,llm_performance,gold_avoid,llm_avoid,gold_additional,llm_additional,diff_fields
1,UQ0005,난 바닐라 향기가 좋은데. 너무 단 건 싫어.,[Vanilla],[Vanilla],"{'season': [], 'daypart': [], 'gender': []}","{'season': [], 'daypart': [], 'gender': []}","{'intensity': '', 'longevity': ''}","{'intensity': '', 'longevity': ''}",[너무 단],[Too sweet],[],[],avoid
4,UQ0015,여름에 뿌릴 향수 추천 좀.,[],[],"{'season': ['summer'], 'daypart': [], 'gender'...","{'season': ['summer'], 'daypart': [], 'gender'...","{'intensity': '', 'longevity': ''}","{'intensity': '', 'longevity': ''}",[],[],[],[여름에 뿌릴 향수 추천 요청],additional
23,UQ0087,가을에 잘 어울리는 우디 향수 추천해줘,[woody],[woody],"{'season': ['autumn'], 'daypart': [], 'gender'...","{'season': ['autumn'], 'daypart': [], 'gender'...","{'intensity': '', 'longevity': ''}","{'intensity': '', 'longevity': ''}",[],[],[],[가을에 잘 어울리는 우디 향수 추천해줘],additional
29,UQ0111,남자가 쓰기 좋은 파우더리 향수 추천해줘,[powdery],[],"{'season': [], 'daypart': [], 'gender': ['male']}","{'season': [], 'daypart': [], 'gender': ['male']}","{'intensity': '', 'longevity': ''}","{'intensity': '', 'longevity': ''}",[],[],[],"[파우더리 향수, 남자가 쓰기 좋은 향수 추천]",scent|additional
32,UQ0140,중성적인 향수에는 뭐가 있을까?,[],[],"{'season': [], 'daypart': [], 'gender': ['neut...","{'season': [], 'daypart': [], 'gender': ['neut...","{'intensity': '', 'longevity': ''}","{'intensity': '', 'longevity': ''}",[],[],[],[중성적인 향수 추천],additional


## 12. Display Avoid QA

이 표는 `LLM Avoid F1 = 0%`가 실제 해석 실패인지 표현 방식 차이인지 사람이 판단하기 위한 자료다. Notebook은 결론을 내리지 않는다.


In [12]:
avoid_columns = ["query_id", "query_text", "gold_avoid", "llm_avoid", "extended_avoid", "avoid_normalized_equal"]
avoid_qa_view = candidates_df.loc[candidates_df["query_id"].isin(avoid_mismatch_ids), avoid_columns]
display(avoid_qa_view)


,query_id,query_text,gold_avoid,llm_avoid,extended_avoid,avoid_normalized_equal
0,UQ0004,"여성스러운 향을 사고 싶은데, 너무 달콤한 건 싫어. 그런 향기가 있을까?",[달콤한],[too sweet],[너무 달콤한],False
1,UQ0005,난 바닐라 향기가 좋은데. 너무 단 건 싫어.,[너무 단],[Too sweet],[너무 단],False
2,UQ0010,"케이크 같으면서도, 너무 느끼하지 않고, 딱 맡았을 때 공주처럼 느껴지는 향수 그런...",[너무 느끼],[],[],False
3,UQ0011,나무 냄새 나는데 너무 절 냄새만 나면 안됨,[절 냄새],[too strong only religious/temple-like scent],[],False
5,UQ0016,여름에 뿌릴 향수 가지고 싶은데 향기 너무 많이 나면 코 아파서... 안 그런거 추...,[향기 너무 많이 나면],[],[],False
6,UQ0020,향수 분사 적당히 되는 거 없나? 입자가 너무 작은 건 싫은데.,[입자가 너무 작은 건],[],[입자가 너무 작은],False
7,UQ0022,나 조말론 블랙베리는 별로더라... 이런 나한테 추천해줄 만한 향수 없나,[조말론 블랙베리],[Blackberry (Jo Malone BlackBerry 계열)],[나 조말론 블랙베리],False
8,UQ0025,소개팅할 때 부담스럽지 않으면서 좋은 인상을 줄 수 있는 향수 추천해줘.,[부담],[],[],False
11,UQ0030,사람 많은 곳에서도 너무 강하게 느껴지지 않는 은은한 향수를 추천해줘.,[강하게],[],[],False
12,UQ0032,향수 냄새가 강하면 머리가 아파서 최대한 부드럽고 편안한 향을 찾고 있어.,[강하면],[],[],False


## 13. Display Additional Sample


In [13]:
additional_columns = ["query_id", "query_text", "gold_additional", "llm_additional", "additional_normalized_equal"]
additional_sample_view = candidates_df.loc[candidates_df["query_id"].isin(additional_sample_ids), additional_columns]
display(additional_sample_view)


,query_id,query_text,gold_additional,llm_additional,additional_normalized_equal
9,UQ0026,조용하고 차분한 사람에게 잘 어울릴 것 같은 향수를 추천해줘.,"[조용, 차분한]",[조용하고 차분한 사람에게 잘 어울리는 향],False
10,UQ0027,지나갈 때 은은하게 좋은 냄새가 나는 정도의 향수를 찾고 있어.,"[지나갈 때 은은, 좋은 냄새]",[지나갈 때 은은하게 좋은 냄새가 나는 정도],False
14,UQ0037,남자친구한테서 났으면 하는 향 찾아줘,[남자친구],[남자친구에게서 났으면 하는 향],False
15,UQ0045,헬스장에서 반응 좋은 향수 추천해줘,"[헬스장, 반응 좋은]",[헬스장에서 반응 좋은 향수 추천],False
16,UQ0050,향수 입문자에게 괜찮은 향수 추천해줘,"[향수 입문자, 괜찮은 향수]",[향수 입문자에게 괜찮은 향수 추천],False
20,UQ0065,은은하게 잔향 좋은 거. 향은 딱히 선호 없음 모름 잘,"[잔향 좋은 거, 선호 없음]","[은은하게 잔향 좋은 거, 향은 딱히 선호 없음/모름]",False
21,UQ0074,비 오는 날에 뿌릴 만한 30대 여성 향수 추천해줘,"[비 오는 날, 30대]",[],False
22,UQ0078,가볍고 부드러운 느낌의 향수,"[가볍고, 부드러운]",[가볍고 부드러운 느낌],False
24,UQ0090,데이트할 때 반응 좋은 향수 추천해줘,"[데이트, 반응 좋은]",[데이트할 때 반응 좋은 향수 추천],False
26,UQ0093,인기많은 남자 향수 추천해줘.,[인기많은 남자],[인기 많은 남자 향수],False


### 사람 판단 예시

예시 1: Gold Avoid가 `['너무 단']`, Prediction이 `['너무 단 향']`이면 `SURFACE_MISMATCH` 후보일 수 있지만 실제 판단은 사람이 한다.

예시 2: 부정 표현이 명시된 Query인데 Gold Avoid가 비어 있고 Prediction에 부정 대상이 있다면 `GOLD_ERROR` 가능성을 사람이 검토한다.

예시 3: Gold Additional이 여러 항목이고 Prediction이 하나의 구절로 묶었다면 Annotation segmentation 일관성 문제인지 사람이 확인한다.


## 14. Save Audit Candidates


In [14]:
JSON_OUTPUT_COLUMNS = [
    "gold_scent", "gold_context", "gold_performance", "gold_avoid", "gold_additional",
    "extended_scent", "extended_context", "extended_performance", "extended_avoid", "extended_additional",
    "llm_scent", "llm_context", "llm_performance", "llm_avoid", "llm_additional",
]
OUTPUT_COLUMNS = [
    "query_id", "query_text", "selection_reason",
    "gold_scent", "gold_context", "gold_performance", "gold_avoid", "gold_additional",
    "extended_scent", "extended_context", "extended_performance", "extended_avoid", "extended_additional",
    "llm_scent", "llm_context", "llm_performance", "llm_avoid", "llm_additional",
    "extended_exact", "llm_exact", "avoid_exact", "additional_exact",
    "avoid_normalized_equal", "additional_normalized_equal",
    "avoid_item_count_gold", "avoid_item_count_llm", "additional_item_count_gold", "additional_item_count_llm",
    "diff_fields", "manual_decision", "manual_issue_type", "manual_comment",
]
audit_output_df = candidates_df[OUTPUT_COLUMNS].copy()
for column in JSON_OUTPUT_COLUMNS:
    audit_output_df[column] = audit_output_df[column].map(lambda value: json.dumps(value, ensure_ascii=False, separators=(",", ":")))
audit_output_df.to_csv(CANDIDATES_PATH, index=False, encoding="utf-8-sig")

saved_candidates_df = pd.read_csv(CANDIDATES_PATH, dtype=str, keep_default_na=False)
if saved_candidates_df.columns.tolist() != OUTPUT_COLUMNS:
    raise RuntimeError("저장된 후보 CSV column 순서 불일치")
if saved_candidates_df["query_id"].duplicated().any() or len(saved_candidates_df) != len(candidates_df):
    raise RuntimeError("저장된 후보 CSV 중복/행 수 오류")
if saved_candidates_df[["manual_decision", "manual_issue_type", "manual_comment"]].ne("").any().any():
    raise RuntimeError("수동 판단 컬럼은 모두 빈 값이어야 합니다.")
print("Saved:", CANDIDATES_PATH.relative_to(PROJECT_ROOT), "rows:", len(saved_candidates_df))


Saved: analysis_outputs\16_golden_set_quality_audit_candidates.csv rows: 60


## 15. Save Audit Guide


In [15]:
overlap_count = int(candidates_df["selection_reason"].str.contains("\|").sum())
guide_text = f"""# Stage 1 Golden Set Quality Audit

## 목적

Golden Set을 재구축하기 위한 작업이 아니라, 일부 낮은 평가가 Annotation 또는 표현 방식 문제에서 발생했는지 빠르게 확인하기 위한 표본 QA다.

## 검토 대상

- LLM_REGRESSED: {len(llm_regressed_ids)}개
- Avoid mismatch: {len(avoid_mismatch_ids)}개
- Additional sample: {len(additional_sample_ids)}개
- 중복 제거 후 최종 QA Query: {len(candidates_df)}개

## 판단 기준

### GOLD_OK

현재 Gold가 Annotation 원칙에 맞고 Prediction이 틀린 경우

### SURFACE_MISMATCH

핵심 의미는 유사해 보이지만 표현 형식 차이 때문에 Strict 평가에서 실패한 경우

### GOLD_ERROR

현재 Gold 자체가 명확하게 잘못된 경우

### UNCERTAIN

판단이 어려운 경우

## manual_issue_type 후보

- NONE
- AVOID_SPAN: 부정 범위 표현 차이
- ADDITIONAL_SEGMENTATION: 표현 분할 방식 차이
- WORDING_VARIATION: 문자열 표현 차이
- SCENT_CANONICALIZATION: 향 표준명 표현 문제
- CONTEXT_LABEL: Gender / Season / Daypart Annotation 문제
- PERFORMANCE_LABEL: Intensity / Longevity Annotation 문제
- OTHER_GOLD_ERROR
- UNCERTAIN

## 중요 원칙

- LLM Prediction을 정답으로 간주하지 않는다.
- 모델 결과에 맞춰 Gold를 수정하지 않는다.
- 명백한 반복 패턴이 발견될 때만 후속 Golden Set 개정 필요성을 검토한다.
- `manual_decision`, `manual_issue_type`, `manual_comment`는 사람이 입력한다.
- 자동 정규화 일치는 lowercase, strip, 중복 제거, 순서 무시만 반영하며 의미 동일 판정이 아니다.

## 검토 순서

1. `selection_reason`과 Query 원문을 확인한다.
2. Gold, Extended, LLM 값을 나란히 비교한다.
3. `manual_decision`에 GOLD_OK / SURFACE_MISMATCH / GOLD_ERROR / UNCERTAIN 중 하나를 입력한다.
4. `manual_issue_type`을 위 후보에서 선택하고 필요하면 `manual_comment`를 작성한다.
"""
GUIDE_PATH.write_text(guide_text, encoding="utf-8")
print("Saved:", GUIDE_PATH.relative_to(PROJECT_ROOT))


Saved: analysis_outputs\16_golden_set_quality_audit_guide.md


### 사람이 검토한 이후를 위한 요약 함수

현재는 모든 수동 판정이 비어 있으므로 성능 또는 Annotation 품질 결론을 만들지 않는다.


In [16]:
MANUAL_DECISIONS = ["GOLD_OK", "SURFACE_MISMATCH", "GOLD_ERROR", "UNCERTAIN"]

def summarize_manual_audit(frame):
    values = frame["manual_decision"].fillna("").astype(str).str.strip().str.upper()
    invalid = sorted(set(values) - set(MANUAL_DECISIONS) - {""})
    if invalid:
        raise ValueError(f"허용되지 않은 manual_decision: {invalid}")
    counts = collections.OrderedDict((label, int(values.eq(label).sum())) for label in MANUAL_DECISIONS)
    counts["UNREVIEWED"] = int(values.eq("").sum())
    total = len(frame)
    return pd.DataFrame({"manual_decision": counts.keys(), "count": counts.values(), "ratio": [count / total if total else np.nan for count in counts.values()]})

display(summarize_manual_audit(saved_candidates_df))
print("현재 수동 판정은 모두 미검토 상태이며 자동 결론을 생성하지 않습니다.")


,manual_decision,count,ratio
0,GOLD_OK,0,0.0
1,SURFACE_MISMATCH,0,0.0
2,GOLD_ERROR,0,0.0
3,UNCERTAIN,0,0.0
4,UNREVIEWED,60,1.0


현재 수동 판정은 모두 미검토 상태이며 자동 결론을 생성하지 않습니다.


## 16. Final QA Summary


In [17]:
input_hashes_after = {label: sha256_file(path) for label, path in INPUT_PATHS.items()}
if input_hashes_before != input_hashes_after:
    raise RuntimeError(f"입력 파일 변경 감지: {[label for label in input_hashes_before if input_hashes_before[label] != input_hashes_after[label]]}")

assert len(llm_regressed_view) == len(llm_regressed_ids)
assert len(avoid_qa_view) == len(avoid_mismatch_ids)
assert len(additional_sample_view) == len(additional_sample_ids) <= ADDITIONAL_SAMPLE_SIZE
assert not (additional_sample_ids & (llm_regressed_ids | avoid_mismatch_ids))
assert candidates_df["query_id"].nunique() == len(candidates_df)
assert saved_candidates_df[["manual_decision", "manual_issue_type", "manual_comment"]].eq("").all().all()

membership_summary = pd.DataFrame({
    "selection_reason": ["LLM_REGRESSED", "AVOID_MISMATCH", "ADDITIONAL_SAMPLE", "Overlap"],
    "query_count": [len(llm_regressed_ids), len(avoid_mismatch_ids), len(additional_sample_ids), overlap_count],
})
exact_reason_summary = candidates_df["selection_reason"].value_counts().rename_axis("exact_selection_reason").reset_index(name="query_count")

print(f"LLM_REGRESSED count: {len(llm_regressed_ids)}")
print(f"Avoid mismatch count: {len(avoid_mismatch_ids)}")
print(f"Additional sample count: {len(additional_sample_ids)}")
print(f"Unique audit queries: {len(candidates_df)}")
display(Markdown("### Selection Reason별 Query 수"))
display(membership_summary)
display(Markdown("### 중복 조합을 포함한 실제 selection_reason"))
display(exact_reason_summary)
print("생성 파일:")
print("-", CANDIDATES_PATH.relative_to(PROJECT_ROOT))
print("-", GUIDE_PATH.relative_to(PROJECT_ROOT))
print("입력 파일 해시 검증: 변경 없음")
print("Golden Set은 수정하지 않았다.")


LLM_REGRESSED count: 5


Avoid mismatch count: 36
Additional sample count: 20
Unique audit queries: 60


### Selection Reason별 Query 수

,selection_reason,query_count
0,LLM_REGRESSED,5
1,AVOID_MISMATCH,36
2,ADDITIONAL_SAMPLE,20
3,Overlap,1


### 중복 조합을 포함한 실제 selection_reason

,exact_selection_reason,query_count
0,AVOID_MISMATCH,35
1,ADDITIONAL_SAMPLE,20
2,LLM_REGRESSED,4
3,LLM_REGRESSED|AVOID_MISMATCH,1


생성 파일:
- analysis_outputs\16_golden_set_quality_audit_candidates.csv
- analysis_outputs\16_golden_set_quality_audit_guide.md
입력 파일 해시 검증: 변경 없음
Golden Set은 수정하지 않았다.
